In [1]:
%load_ext autoreload
%autoreload 2

## 1. Imports

In [2]:
import os
import sys

sys.path.append("..")

import numpy as np
import torch
from tqdm import tqdm

import wandb
from src.models.light_gcot import LightGCOT
import torch.nn as nn

In [3]:
device = torch.device(f"cuda:{torch.cuda.current_device()}" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [4]:
torch.set_default_device(device)

## 2. Dataset and Samplers

In [5]:
from src.utils.datasets import get_Splatter_dataset, get_Splatter_loaders_and_samplers

In [6]:
data_set = get_Splatter_dataset()

Dataset path: ../datasets/Splatter


In [7]:
BATCH_SIZE = 128
NUM_LABELED = 10
TRAIN_SUBSET_SIZE = 2

In [8]:
loader_kwargs = {"num_workers": 0, "pin_memory": True, "generator": torch.Generator(device="cuda")}

In [9]:
source_loader, target_loader, target_test_loader, train_XY_sampler, full_XY_sampler = (
    get_Splatter_loaders_and_samplers(
        data_set, BATCH_SIZE, NUM_LABELED, TRAIN_SUBSET_SIZE, loader_kwargs=loader_kwargs
    )
)

Initial source labels: [ 0  1  2  3  4  5  6  7  8  9 10]
Initial target labels: [0 1 3 4 5 6 7 8]
Common labels: [0 1 3 4 5 6 7 8]
Label mapping: {0: 0, 1: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7}
Processed source labels: [0 1 2 3 4 5 6 7]
Processed target labels: [0 1 3 4 5 6 7 8]
Source dataset shape: (3329, 657)
Target dataset shape: (2108, 657)


## 3. Config

In [10]:
X_DIM = data_set["features"].shape[1]
Y_DIM = data_set["features"].shape[1]
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 10
M_POTENTIALS = 10
EPSILON = 0.002
INIT_BY_SAMPLES = False
A_DIAGONAL_INIT = 0.5

SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

PLOT_EVERY = 1000
MAX_STEPS = 20000
CONTINUE = -1

In [11]:
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)

In [12]:
EXP_COST = "MLP"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"LightGCOT_Batch_Effect_EPSILON_{EPSILON}_MAX_STEPS_{MAX_STEPS}_N_{N_POTENTIALS}_M_{M_POTENTIALS}_with_{EXP_COST}_cost_included_{EXP_COST_INCLUDED}_N_PAIRED_{NUM_LABELED}_M_UNPAIRED_{len(train_XY_sampler.dataset)}"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    INIT_BY_SAMPLES=INIT_BY_SAMPLES,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=NUM_LABELED,
    M_UNPAIRED_SAMPLES=len(train_XY_sampler.dataset),
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

## 4. Classifier Pretrain

In [13]:
class Classifier(nn.Module):
    def __init__(self, input_size: int, output_size: int):
        super(Classifier, self).__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, x):
        x = self.fc(x)
        return x

In [14]:
# TODO: fix 8
classifier = Classifier(X_DIM, 8).cuda()
classifier_opt = torch.optim.Adam(classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

In [15]:
# TODO: fix
total_step = len(target_loader)
for epoch in tqdm(range(30)):
    for i, (inputs, labels) in enumerate(target_loader):
        inputs = inputs.reshape(-1, 657)
        outputs = classifier(inputs.cuda())
        loss = criterion(outputs, labels.cuda())
        classifier_opt.zero_grad()
        loss.backward()
        classifier_opt.step()

        if (i+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{30}], Step [{i+1}/{total_step}], Loss: {loss.item():.4f}")
            with torch.no_grad():
                correct = 0
                total = 0
                for inputs, labels in target_test_loader:
                    inputs = inputs.reshape(-1, 657)
                    outputs = classifier(inputs.cuda())
                    _, predicted = torch.max(outputs.data, 1)
                    total += labels.size(0)
                    correct += (predicted == labels.cuda()).sum().item()

                print(f"Accuracy of the model on the test inputs: {100 * correct / total}%")

 10%|███████████▎                                                                                                     | 3/30 [00:00<00:03,  7.26it/s]

Epoch [1/30], Step [10/16], Loss: 1.9005
Accuracy of the model on the test inputs: 24.240986717267553%
Epoch [2/30], Step [10/16], Loss: 1.4788
Accuracy of the model on the test inputs: 55.83491461100569%
Epoch [3/30], Step [10/16], Loss: 1.2595
Accuracy of the model on the test inputs: 65.03795066413662%


 23%|██████████████████████████▎                                                                                      | 7/30 [00:00<00:01, 12.71it/s]

Epoch [4/30], Step [10/16], Loss: 1.1555
Accuracy of the model on the test inputs: 68.78557874762808%
Epoch [5/30], Step [10/16], Loss: 0.9998
Accuracy of the model on the test inputs: 72.10626185958255%
Epoch [6/30], Step [10/16], Loss: 0.8763
Accuracy of the model on the test inputs: 74.05123339658444%
Epoch [7/30], Step [10/16], Loss: 0.7867
Accuracy of the model on the test inputs: 75.99620493358634%


 30%|█████████████████████████████████▉                                                                               | 9/30 [00:00<00:01, 14.22it/s]

...

Epoch [20/30], Step [10/16], Loss: 0.3841
Accuracy of the model on the test inputs: 94.44971537001898%
Epoch [21/30], Step [10/16], Loss: 0.3244
Accuracy of the model on the test inputs: 95.20872865275142%
Epoch [22/30], Step [10/16], Loss: 0.2916
Accuracy of the model on the test inputs: 95.87286527514232%
Epoch [23/30], Step [10/16], Loss: 0.2897
Accuracy of the model on the test inputs: 96.15749525616698%


 83%|█████████████████████████████████████████████████████████████████████████████████████████████▎                  | 25/30 [00:01<00:00, 17.65it/s]

Epoch [24/30], Step [10/16], Loss: 0.2566
Accuracy of the model on the test inputs: 96.86907020872866%
Epoch [25/30], Step [10/16], Loss: 0.2383
Accuracy of the model on the test inputs: 97.2011385199241%
Epoch [26/30], Step [10/16], Loss: 0.2813
Accuracy of the model on the test inputs: 97.6280834914611%
Epoch [27/30], Step [10/16], Loss: 0.2493
Accuracy of the model on the test inputs: 97.81783681214421%


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [00:01<00:00, 15.01it/s]

Epoch [28/30], Step [10/16], Loss: 0.2279
Accuracy of the model on the test inputs: 98.14990512333966%
Epoch [29/30], Step [10/16], Loss: 0.2207
Accuracy of the model on the test inputs: 98.24478178368122%
Epoch [30/30], Step [10/16], Loss: 0.2441
Accuracy of the model on the test inputs: 98.33965844402277%


## 5. Model initialization

In [16]:
D = LightGCOT(
    x_dim=X_DIM,
    y_dim=Y_DIM,
    n_potentials=N_POTENTIALS,
    m_potentials=M_POTENTIALS,
    epsilon=EPSILON,
    sampling_batch_size=SAMPLING_BATCH_SIZE,
    A_diagonal_init=A_DIAGONAL_INIT,
    cost_function=EXP_COST,
)

if INIT_BY_SAMPLES:
    D.init_a_by_samples(Y_sampler.sample(N_POTENTIALS))

D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

if CONTINUE > -1:
    D_opt.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_{OUTPUT_SEED}_{CONTINUE}.pt")))

## Classifier

In [17]:
def test_accuracy(classifier, loader, T=None, log=False):
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in loader:
            inputs = inputs.reshape(-1, 657)
            if T:
                inputs = T(inputs.cuda())
            outputs = classifier(inputs.cuda())
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels.cuda()).sum().item()

        accuracy = 100 * correct / total
        if not log:
            print(f"Accuracy of the model on the test inputs: {accuracy}%")
            return accuracy
        else:
            return {f"Accuracy on the test": accuracy}

## 5. Model training

In [18]:
from src.utils.plotting.matplotlib import plot_A_parameters

In [20]:
wandb.init(name=EXP_NAME, config=config)

for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
    # training loop
    D_opt.zero_grad()

    _X, _Y = full_XY_sampler.sample(BATCH_SIZE)
    X, Y = _X.flatten(start_dim=0, end_dim=1), _Y.flatten(start_dim=0, end_dim=1)

    log_v_m = D.compute_log_v_m(X)  # [bs x M]
    b_m = D.compute_b_m(X)  # [bs x M x y_dim]

    log_w_n = D.compute_log_w_n()  # [N]
    a_n = D.compute_a_n()  # [N x y_dim]
    A_n = D.compute_A_n()  # [N x y_dim]

    f_c = D.compute_dual_potential(log_w_n, a_n, A_n, log_v_m, b_m)
    f = D.compute_primal_potential(Y, log_w_n, a_n, A_n)

    if EXP_COST_INCLUDED:
        _X_paired, _Y_paired = train_XY_sampler.sample()
        X_paired = _X_paired.flatten(start_dim=0, end_dim=1)
        Y_paired = _Y_paired.flatten(start_dim=0, end_dim=1)
        log_v_m_paired = D.compute_log_v_m(X_paired)  # [bs x M]
        b_m_paired = D.compute_b_m(X_paired)  # [bs x M x y_dim]

        c = D.compute_cost(Y_paired, log_v_m_paired, b_m_paired)
        D_loss = c.mean() - (f_c + f).mean()
        D_loss.backward()
        wandb.log({r"$c(x, y)$": c.mean().item()}, step=step)
    else:
        D_loss = -(f_c + f).mean()
        D_loss.backward()
    D_gradient_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
    D_opt.step()

    wandb.log({f"D gradient norm": D_gradient_norm.item()}, step=step)
    wandb.log({f"D_loss": D_loss.item()}, step=step)
    wandb.log({r"$-f^c(x)$": -f_c.mean().item()}, step=step)
    wandb.log({r"$-f(y)$": -f.mean().item()}, step=step)
    wandb.log({r"$-f(y)-f^c(x)$": -(f_c + f).mean().item()}, step=step)
    wandb.log({f"lam_min(A_n)": torch.min(A_n)}, step=step)
    wandb.log({f"lam_max(A_n)": torch.max(A_n)}, step=step)

    if step % PLOT_EVERY == 0:
        accuracy_dict = test_accuracy(classifier, source_loader, D, log=True)
        wandb.log(accuracy_dict, step=step)

        A_dict = plot_A_parameters(D, log=True)
        # B_dict = plot_B_parameters(D, starting_points, log=True)
        # distr_dict = plot_distributions(D, X_sampler, Y_sampler, X_paired, Y_paired, starting_points, log=True)
        wandb.log(A_dict)# | B_dict)# | distr_dict)

        torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{step}.pt"))
        torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_{step}.pt"))

torch.save(D.state_dict(), os.path.join(OUTPUT_PATH, f"D_{MAX_STEPS}.pt"))
torch.save(D_opt.state_dict(), os.path.join(OUTPUT_PATH, f"D_opt_{MAX_STEPS}.pt"))

wandb.finish()

 47%|█████████████████████████████████████████████████▊                                                         | 9302/20000 [04:08<04:39, 38.25it/s]

## Plotting

In [ ]:
plot_distributions(D, X_sampler, Y_sampler, starting_points)